<a href="https://colab.research.google.com/github/bmulaw/NLP-Notebooks/blob/main/NLP_TP3_Fine_Tuning_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP TP3 — Full fine-tuning and LoRA

**Language:** English · **Environment:** Google Colab · **Suggested duration:** 3 hours

In TP2, you used pretrained models as they were, for inference only. In this lab you will train them further on labelled examples, measure their predictions before and after training, and compare two ways of training: full fine-tuning and LoRA.

| Time | Work |
|---|---|
| 20 min | Setup, datasets, baseline evaluation |
| 35 min | Guided full fine-tuning for IMDb sentiment |
| 25 min | How full fine-tuning and LoRA work; guided IMDb LoRA |
| 45 min | Guided named-entity recognition: baseline, full fine-tuning, LoRA |
| 55 min | Autonomous emotion classification and comparison |

### Before you start

In Colab choose **Runtime → Change runtime type → T4 GPU**. The lab also runs on a CPU, but each training run then takes one to two minutes. Models and datasets are downloaded the first time they are used. Run the cells in order, from top to bottom, and save the notebook with your results and written answers.

The samples are deliberately small so that every run fits in the session. The scores you obtain are useful for comparing methods during the lab. A real evaluation would need much more data.

### Vocabulary used in this lab

- **Checkpoint:** a saved model (its configuration and its weights) that we download from the Hugging Face Hub by name.
- **Parameters** or **weights:** the numbers inside the model. Training changes them.
- **Head:** the last layer of the model, which turns the internal vectors into the scores of each class. A sentiment head has 2 outputs; an NER head has one output per entity tag.
- **Logits:** the raw scores produced by the head, one per class. The predicted class is the one with the largest logit.
- **Fine-tuning:** training an already trained model a little more, on labelled examples of the task we care about.
- **Baseline:** in this lab, the checkpoint exactly as downloaded, evaluated before we train it.

In [ ]:
# Installs the library versions this notebook was tested with (takes about a minute).
# If Colab shows a "Restart session" button afterwards, click it, then continue with the next cell.
%pip install --only-binary=:all: "transformers==4.46.3" "peft==0.13.2" "datasets==4.3.0" "accelerate==1.1.1" "scikit-learn" "safetensors"
%pip install --no-deps "seqeval==1.2.2"

In [2]:
import gc
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from seqeval.metrics import precision_score as entity_precision
from seqeval.metrics import recall_score as entity_recall
from seqeval.metrics import f1_score as entity_f1
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForTokenClassification, DataCollatorWithPadding,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments,
                          set_seed)
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
set_seed(SEED)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found: the lab will work, but training will be slow.")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


## 1. Experimental protocol — guided

Each dataset is divided into two groups of examples:

- the **training** examples, which the model learns from: their labels are used to update the parameters;
- the **validation** examples, which are only used to measure the model. The model never learns from them.

For one task, we compare three variants: **the baseline, full fine-tuning and LoRA**. To make the comparison fair, all three start from the same checkpoint, use the same preprocessing and are measured on the same validation examples.

We use the validation examples to compare the variants during the lab. If you then used these results to choose a method or a learning rate, you would need a third group of examples, the **test set**, kept untouched until the very end, to report an unbiased final score.

### Metrics

All scores are proportions between 0 and 1 (multiply by 100 to read a percentage).

- **Accuracy:** the share of validation examples whose predicted class is correct.
- **Precision** of a class: among the examples the model assigned to this class, the share that really belong to it.
- **Recall** of a class: among the examples that really belong to this class, the share the model found.
- **F1:** a single score combining precision and recall (their harmonic mean). It is high only when both are high.
- **Macro F1:** the F1 of each class, averaged with the same weight for every class. A rare class therefore counts as much as a frequent one. If a class has very few validation examples, its F1 changes a lot from one sample to another, and macro F1 becomes less stable.

For named-entity recognition (NER, section 4), we measure precision, recall and F1 on **whole entities**: a predicted entity counts as correct only if its type and its first and last words both match the reference.

### What "baseline" means here

The checkpoints we download have already been fine-tuned by their authors and already have a head for the task. The baseline is therefore a model that already works. We evaluate it as downloaded, before training it on our own training sample. For example, the sentiment checkpoint was fine-tuned on SST-2, a dataset of short movie-review sentences, and we evaluate it on IMDb, a different dataset of full reviews.

In [3]:
# Settings shared by every experiment.
# If you change one of them, change it before section 2 and rerun everything,
# so that the three variants of a task always use the same settings.
TRAIN_SIZE = 400   # training examples per task
VALID_SIZE = 200   # validation examples per task
MAX_LENGTH = 128   # longer inputs are cut after 128 tokens
EPOCHS = 1         # number of passes over the training examples
BATCH_SIZE = 8     # examples processed together at each training step

def sample_split(dataset, count):
    """Select a reproducible random sample of at most count examples."""
    return dataset.shuffle(seed=SEED).select(range(min(count, len(dataset))))

def classification_metrics(prediction):
    """Compute accuracy and macro F1 from the model's logits and the reference labels."""
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predicted),
            "macro_f1": f1_score(labels, predicted, average="macro", zero_division=0)}

def parameter_counts(model):
    """Count all parameters and those that training is allowed to change."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def make_arguments(name, learning_rate):
    """Configure a small, reproducible run that saves no model files."""
    return TrainingArguments(
        output_dir=f"./tp3_runs/{name}", learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, weight_decay=0.01, seed=SEED, data_seed=SEED,
        save_strategy="no", eval_strategy="no", logging_strategy="no",
        report_to="none", fp16=False, dataloader_num_workers=0,
        label_names=["labels"])

def run_experiment(name, model, tokenizer, training_data, validation_data,
                   collator, metrics_function, learning_rate=2e-5, train=False):
    """Optionally train the model, then evaluate it on the validation examples.

    Returns one row of results: parameter counts, training time,
    peak GPU memory and validation scores.
    """
    set_seed(SEED)
    total, trainable = parameter_counts(model)
    trainer = Trainer(model=model, args=make_arguments(name, learning_rate),
                      train_dataset=training_data, eval_dataset=validation_data,
                      data_collator=collator, compute_metrics=metrics_function)
    seconds, peak_gib = None, None
    if train:
        # The GPU works asynchronously: synchronizing makes the timer wait for it.
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        trainer.train()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            peak_gib = torch.cuda.max_memory_allocated() / (1024 ** 3)
        seconds = time.perf_counter() - start
    # Evaluation happens after the timer has stopped.
    result = trainer.evaluate()
    row = {"variant": name, "total_parameters": total, "trainable_parameters": trainable,
           "train_seconds": seconds, "peak_gpu_gib": peak_gib}
    row.update({key.removeprefix("eval_"): value for key, value in result.items()
                if key in ("eval_accuracy", "eval_macro_f1", "eval_precision", "eval_recall", "eval_entity_f1")})
    print(row)
    # Free the memory used by this model before the next experiment loads another one.
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

### Reading the results of `run_experiment`

- `trainable_parameters` counts the parameters that the optimizer would be allowed to change. For the baseline, it equals `total_parameters` even though nothing is trained: the baseline row only tells you how many parameters full fine-tuning updates.
- `train_seconds` is the duration of training alone. Downloads, tokenization and evaluation happen outside this measurement. For the baseline, nothing is trained, so the value is empty (`None`).
- `peak_gpu_gib` is the largest amount of GPU memory that PyTorch used during training, in GiB. It counts the model weights and everything training needs. Colab's resource monitor can show a larger figure, because it also counts memory reserved by other parts of the session. On a CPU the value is empty.
- One run is a single measurement. If LoRA is faster in your table, this shows it for this model, these settings and this GPU only.

Each experiment loads a **fresh copy** of the checkpoint. The LoRA run therefore starts from the original model, and it cannot benefit from the full fine-tuning done just before.

## 2. IMDb sentiment — baseline and full fine-tuning (guided)

IMDb contains English movie reviews labelled `0 = negative` and `1 = positive`. The DistilBERT checkpoint we use was fine-tuned on SST-2 and already predicts these two labels. We first evaluate it on IMDb as it is (the baseline). Then we train all its parameters on IMDb training reviews (full fine-tuning).

IMDb has a training split and a test split. We take 1,000 reviews out of the training split to form a validation pool, and draw our 200 validation reviews from this pool. The 400 training reviews come from the rest of the training split, so no review is used both for training and for validation. We leave the test split aside.

Reviews are often long. To keep the runs fast, the model only reads the first 128 tokens of each review (about 100 words), for all three variants. The next cell counts how many reviews are cut: you will see that most of them are, so the model often decides from the beginning of a review only. The cell may also print a warning saying that a sequence is longer than the maximum length (512 tokens for this model). This warning comes from the counting step, which tokenizes complete reviews, and can be ignored.

In [5]:
IMDB_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english" # pre-trained baseline model
imdb = load_dataset("stanfordnlp/imdb", split="train")
imdb_parts = imdb.train_test_split(test_size=1000, seed=SEED, stratify_by_column="label")
imdb_train = sample_split(imdb_parts["train"], TRAIN_SIZE)
imdb_val = sample_split(imdb_parts["test"], VALID_SIZE)
imdb_tokenizer = AutoTokenizer.from_pretrained(IMDB_MODEL)
print("Training reviews:", len(imdb_train), "Validation reviews:", len(imdb_val))
print("Labels:", imdb.features["label"])
print("Example:", imdb_train[0]["text"][:300], "| label:", imdb_train[0]["label"])

def tokenize_reviews(batch):
    """Convert reviews to token IDs, keeping at most MAX_LENGTH tokens (special tokens included)."""
    return imdb_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

def count_truncated(dataset, tokenizer):
    """Count the reviews that contain more than MAX_LENGTH tokens."""
    # dataset["text"] is a lazy column object: list() gives the tokenizer a plain list of strings.
    lengths = tokenizer(list(dataset["text"]), truncation=False, return_length=True)["length"]
    return sum(length > MAX_LENGTH for length in lengths)

print("Truncated training reviews:", count_truncated(imdb_train, imdb_tokenizer), "/", len(imdb_train))
print("Truncated validation reviews:", count_truncated(imdb_val, imdb_tokenizer), "/", len(imdb_val))
imdb_train_tok = imdb_train.map(tokenize_reviews, batched=True, remove_columns=["text"])
imdb_val_tok = imdb_val.map(tokenize_reviews, batched=True, remove_columns=["text"])
imdb_collator = DataCollatorWithPadding(tokenizer=imdb_tokenizer)

Training reviews: 400 Validation reviews: 200
Labels: ClassLabel(names=['neg', 'pos'])
Example: The most ridiculous thing about this ridiculous movie is its conceit that if one becomes a saint, he or she and his or her family and his or her significant other live forever. Let's forget that in order to become a saint, the saint must be dead, and saints don't have significant others. That, for a | label: 0


Token indices sequence length is longer than the specified maximum sequence length for this model (852 > 512). Running this sequence through the model will result in indexing errors


Truncated training reviews: 362 / 400
Truncated validation reviews: 175 / 200


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

### 2A. Evaluate the baseline

The model processes reviews in small groups called **batches**. Reviews in a batch have different lengths, so the **collator** adds padding tokens to give them the same length. `Trainer.evaluate()` runs the model on the validation reviews and compares its predictions with the labels. It does not change the model.

In [6]:
imdb_results = []
imdb_results.append(run_experiment(
    "imdb_baseline", AutoModelForSequenceClassification.from_pretrained(IMDB_MODEL),
    imdb_tokenizer, imdb_train_tok, imdb_val_tok, imdb_collator, classification_metrics))

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

{'variant': 'imdb_baseline', 'total_parameters': 66955010, 'trainable_parameters': 66955010, 'train_seconds': None, 'peak_gpu_gib': None, 'accuracy': 0.8, 'macro_f1': 0.7987117552334944}


### 2B. Full fine-tuning

`Trainer.train()` repeats the same steps for each batch of training reviews:

1. the model predicts a class for each review;
2. the **loss** (here, cross-entropy) measures how far the predictions are from the labels: it is small when the model gives a high score to the correct class;
3. **backpropagation** computes, for every trainable parameter, the direction in which it should move to reduce the loss (its gradient);
4. the **optimizer** moves each trainable parameter a small step in that direction.

In full fine-tuning, every parameter of the model is trainable. One **epoch** is one pass over all the training reviews: with 400 reviews and batches of 8, that makes 50 steps. The **learning rate** sets the size of each step. `2e-5` (0.00002) is a usual value for full fine-tuning of this kind of model: larger steps could destroy what the model has already learned.

The baseline is already good at sentiment, and we train on few examples for a single epoch. Expect a small change, which can be an improvement or a degradation.

In [7]:
imdb_results.append(run_experiment(
    "imdb_full", AutoModelForSequenceClassification.from_pretrained(IMDB_MODEL),
    imdb_tokenizer, imdb_train_tok, imdb_val_tok, imdb_collator, classification_metrics,
    learning_rate=2e-5, train=True))
display(pd.DataFrame(imdb_results))

Step,Training Loss


{'variant': 'imdb_full', 'total_parameters': 66955010, 'trainable_parameters': 66955010, 'train_seconds': 5.904064414000004, 'peak_gpu_gib': 1.2866230010986328, 'accuracy': 0.775, 'macro_f1': 0.7749493636068115}


,variant,total_parameters,trainable_parameters,train_seconds,peak_gpu_gib,accuracy,macro_f1
0,imdb_baseline,66955010,66955010,NaN,NaN,0.800,0.798712
1,imdb_full,66955010,66955010,5.904064,1.286623,0.775,0.774949


**Questions:** Did validation accuracy and macro F1 improve? How many parameters were trainable? With 200 validation reviews, one review represents 0.005 of accuracy: is the difference you observe large compared with this? Does this result prove that the model will be better on all future reviews?

- **No, validation accuracy and macro F1 did not improve. All 66,955,010 parameters were trainable. Yes, the difference in accuracy between the model and the finetuning was relatively large compared to the 0.005 accuracy per each review (the difference being 0.025).**

## 3. Why LoRA? — guided

During full fine-tuning, the GPU must hold, for every trainable parameter, the parameter itself, its gradient and two extra values kept by the optimizer (**Adam** keeps a running average of past gradients and of their squares). It must also keep the intermediate results of the forward pass (the **activations**), which backpropagation reuses. For large models, this quickly exceeds the memory of a GPU. The exact amounts depend on the architecture, the number format, the sequence length and the batch size.

**LoRA** (Low-Rank Adaptation) reduces the number of trainable parameters. It freezes the original weight matrix `W` of selected layers and learns a small correction added to it:

`W_effective = W + (alpha / r) · B · A`

- `W` has shape `d_out × d_in` and stays frozen;
- `A` has shape `r × d_in` and `B` has shape `d_out × r`; only `A` and `B` are trained;
- the **rank** `r` is much smaller than `d_in` and `d_out`;
- `alpha / r` is a fixed scaling factor applied to the correction.

**Example.** In DistilBERT, the query projection `q_lin` of an attention layer is a `768 × 768` matrix: 589,824 weights. With `r = 8`, LoRA trains `A` (8 × 768) and `B` (768 × 8): 8 × (768 + 768) = 12,288 values, about 2 % of the matrix.

`B` starts at zero, so at the beginning of training the correction is zero and the LoRA model makes exactly the same predictions as the checkpoint. Training then learns the correction.

Some points to keep in mind:

- The frozen base model is still loaded in memory, and the forward and backward passes still go through the whole network. On a small model like DistilBERT, the savings in time and memory can therefore be small. They grow with the size of the model.
- PEFT, the library that implements LoRA, also keeps the head trainable, because it is specific to the task. In DistilBERT the head has two layers, `pre_classifier` (768 × 768) and `classifier` (768 × 2). For IMDb this gives 739,586 trainable values: 147,456 in the LoRA matrices (2 matrices × 6 layers × 12,288), 590,592 in `pre_classifier` and 1,538 in `classifier`. Most of them belong to the head.
- PEFT adds the adapters and a trainable copy of the head next to the frozen original. `total_parameters` is therefore slightly larger for LoRA than for the other variants.
- LoRA usually needs a larger learning rate than full fine-tuning, because it trains few parameters that start from zero. We use `2e-4`, ten times the full fine-tuning value.

The LoRA settings are: `r` sets the rank, `lora_alpha` sets the scaling, `lora_dropout` randomly ignores part of the adapter during training to limit overfitting, and `target_modules` lists the layers that receive an adapter. We adapt the query and value projections of attention. Their names depend on the architecture: `q_lin` and `v_lin` in DistilBERT, `query` and `value` in the BERT model of section 4. `TaskType` tells PEFT which kind of model it is wrapping (sequence classification here), so that it handles the head correctly.

In [8]:
def make_lora(checkpoint, model_class, task_type, target_modules):
    """Load a fresh copy of the checkpoint and add LoRA adapters to the chosen layers."""
    set_seed(SEED)
    model = model_class.from_pretrained(checkpoint)
    config = LoraConfig(task_type=task_type, r=8, lora_alpha=16,
                        lora_dropout=0.1, target_modules=target_modules,
                        bias="none")
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    return model

imdb_results.append(run_experiment(
    "imdb_lora", make_lora(IMDB_MODEL, AutoModelForSequenceClassification,
                             TaskType.SEQ_CLS, ["q_lin", "v_lin"]),
    imdb_tokenizer, imdb_train_tok, imdb_val_tok, imdb_collator, classification_metrics,
    learning_rate=2e-4, train=True))
display(pd.DataFrame(imdb_results))

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


Step,Training Loss


{'variant': 'imdb_lora', 'total_parameters': 67694596, 'trainable_parameters': 739586, 'train_seconds': 4.164845868999919, 'peak_gpu_gib': 0.5250082015991211, 'accuracy': 0.8, 'macro_f1': 0.7990151743543362}


,variant,total_parameters,trainable_parameters,train_seconds,peak_gpu_gib,accuracy,macro_f1
0,imdb_baseline,66955010,66955010,NaN,NaN,0.800,0.798712
1,imdb_full,66955010,66955010,5.904064,1.286623,0.775,0.774949
2,imdb_lora,67694596,739586,4.164846,0.525008,0.800,0.799015


In [10]:
imdb_results.append(run_experiment(
    "imdb_lora_lr2e-5", make_lora(IMDB_MODEL, AutoModelForSequenceClassification,
                             TaskType.SEQ_CLS, ["q_lin", "v_lin"]),
    imdb_tokenizer, imdb_train_tok, imdb_val_tok, imdb_collator, classification_metrics,
    learning_rate=2e-5, train=True))
display(pd.DataFrame(imdb_results))

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


Step,Training Loss


{'variant': 'imdb_lora_lr2e-5', 'total_parameters': 67694596, 'trainable_parameters': 739586, 'train_seconds': 3.5115375300001688, 'peak_gpu_gib': 0.5250082015991211, 'accuracy': 0.8, 'macro_f1': 0.7987117552334944}


,variant,total_parameters,trainable_parameters,train_seconds,peak_gpu_gib,accuracy,macro_f1
0,imdb_baseline,66955010,66955010,NaN,NaN,0.800,0.798712
1,imdb_full,66955010,66955010,5.904064,1.286623,0.775,0.774949
2,imdb_lora,67694596,739586,4.164846,0.525008,0.800,0.799015
3,imdb_lora,67694596,739586,3.804662,0.525008,0.800,0.798712
4,imdb_lora_lr2e-5,67694596,739586,3.511538,0.525008,0.800,0.798712


**Questions:** Compare trainable parameters, training time, peak GPU memory, accuracy and macro F1 across the three rows. The full and LoRA runs use different learning rates, each usual for its method. Can you then attribute a difference in score to the number of trainable parameters alone? What would you change to make the comparison more convincing?

- **The trainable parameters shrunk from 66,955,010 parameters to 739,586 parameters, the training time reduced by about 1.8 seconds (~31% reduction in time), the peak GPU memory went from 1.29 to 0.53, and accuracy has suprisingly increased under LoRA (matching the baseline model) as well as the macro F1 exceeding the baseline model score. While yes the learning rate is different in LoRA, I've ran the same LoRA model with the same learning rate as the full finetuning and I see a relatively similar peak GPU GiB, accuracy, and macro F1. The only major difference is in the training time (from 4.16sec in LoRA with learning rate 2e-4 to 3.80/3.51sec in LoRA with learning rate 2e-5 ran twice). I think, however, you can attribute the difference in score between the full-finetuning and the LoRA finetuning to the number of trainable parameters alone.**

## 4. Named-entity recognition — guided

**Named-entity recognition (NER)** finds names in a text and gives each one a type: person (`PER`), organization (`ORG`) or location (`LOC`). Unlike sentiment, it predicts one label **per word** (token classification).

IMDb gives a sentiment label for each review but does not mark the names it contains, so we cannot use it to train or evaluate NER. We use the English part of **WikiAnn**, a dataset of sentences from Wikipedia whose names are labelled with **BIO tags**:

- `B-PER` marks the first word of a person's name (B for *beginning*);
- `I-PER` marks the following words of the same name (I for *inside*);
- `O` marks a word that is outside any entity.

For example, `Barack/B-PER Obama/I-PER visited/O Paris/B-LOC`.

The checkpoint `dslim/bert-base-NER` was trained on CoNLL-2003, a dataset of news articles. WikiAnn comes from another source and was annotated with its own conventions. The checkpoint also knows a fourth type, `MISC` (miscellaneous names), which does not exist in WikiAnn: every `MISC` it predicts counts as an error here.

As before, we compare the baseline, full fine-tuning and LoRA on the same WikiAnn samples. Two preparation steps are needed:

1. **Label IDs.** The model and the dataset both number their tags, but with different numbers. We convert each dataset tag to the model's number for the same tag name.
2. **Subwords.** The tokenizer can split a word into several pieces: for instance, a rare name like `Kowalczyk` can become `Ko`, `##wal`, `##cz`, `##yk` (`##` marks a piece that continues the previous one). The dataset gives one tag per word, so we put the tag on the first piece only. The other pieces and the special tokens receive the value `-100`, which the loss ignores. Evaluation also skips these positions.

Sentences longer than 128 tokens are cut, and any entity after the cut is lost. This is rare with WikiAnn's short sentences.

In [11]:
NER_MODEL = "dslim/bert-base-NER"
wiki = load_dataset("unimelb-nlp/wikiann", "en")
ner_train = sample_split(wiki["train"], TRAIN_SIZE)
ner_val = sample_split(wiki["validation"], VALID_SIZE)
ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL, use_fast=True)
reference_names = wiki["train"].features["ner_tags"].feature.names
probe = AutoModelForTokenClassification.from_pretrained(NER_MODEL)
id_to_name = {int(k): v for k, v in probe.config.id2label.items()}
name_to_id = {name: idx for idx, name in id_to_name.items()}
print("Dataset labels:", reference_names)
print("Checkpoint labels:", id_to_name)
print("Example tokens:", ner_train[0]["tokens"])
print("Example tags:", [reference_names[i] for i in ner_train[0]["ner_tags"]])
# Every dataset tag must exist in the model, otherwise the conversion would fail.
assert all(name in name_to_id for name in reference_names)
del probe
gc.collect()

def tokenize_and_align(batch):
    """Tokenize the words and give each word's tag to its first subword; -100 elsewhere."""
    encoded = ner_tokenizer(batch["tokens"], is_split_into_words=True,
                            truncation=True, max_length=MAX_LENGTH)
    aligned = []
    for row, tags in enumerate(batch["ner_tags"]):
        previous_word = None
        labels = []
        # word_ids gives, for each token, the index of the word it comes from (None for special tokens).
        for word_id in encoded.word_ids(batch_index=row):
            if word_id is None or word_id == previous_word:
                labels.append(-100)
            else:
                labels.append(name_to_id[reference_names[tags[word_id]]])
            previous_word = word_id
        aligned.append(labels)
    encoded["labels"] = aligned
    return encoded

ner_train_tok = ner_train.map(tokenize_and_align, batched=True, remove_columns=ner_train.column_names)
ner_val_tok = ner_val.map(tokenize_and_align, batched=True, remove_columns=ner_val.column_names)
ner_collator = DataCollatorForTokenClassification(ner_tokenizer)

example = ner_train_tok[0]
print("Subword tokens:", ner_tokenizer.convert_ids_to_tokens(example["input_ids"]))
print("Aligned labels:", [id_to_name[l] if l != -100 else -100 for l in example["labels"]])

def ner_metrics(prediction):
    """Measure whole-entity precision, recall and F1, skipping positions labelled -100."""
    logits, labels = prediction
    predicted = np.argmax(logits, axis=-1)
    true_names, predicted_names = [], []
    for gold_row, pred_row in zip(labels, predicted):
        true_names.append([id_to_name[int(g)] for g in gold_row if g != -100])
        predicted_names.append([id_to_name[int(p)] for g, p in zip(gold_row, pred_row) if g != -100])
    return {"precision": entity_precision(true_names, predicted_names, zero_division=0),
            "recall": entity_recall(true_names, predicted_names, zero_division=0),
            "entity_f1": entity_f1(true_names, predicted_names, zero_division=0)}

README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

... (8 outputs left out)

Dataset labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Checkpoint labels: {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}
Example tokens: ["''", 'January', '21', "''", '–', 'Nanny', 'and', 'the', 'Professor']
Example tags: ['O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'I-PER', 'I-PER']


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Subword tokens: ['[CLS]', "'", "'", 'January', '21', "'", "'", '–', 'Nan', '##ny', 'and', 'the', 'Professor', '[SEP]']
Aligned labels: [-100, 'O', -100, 'O', 'O', 'O', -100, 'O', 'B-PER', -100, 'I-PER', 'I-PER', 'I-PER', -100]


Look at the two lines printed at the end: each subword token is shown with the label it will be trained on. Check that only the first piece of each word carries a tag.

### 4A. Evaluate the original NER model

In [12]:
ner_results = []
ner_results.append(run_experiment(
    "ner_baseline", AutoModelForTokenClassification.from_pretrained(NER_MODEL),
    ner_tokenizer, ner_train_tok, ner_val_tok, ner_collator, ner_metrics))

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


{'variant': 'ner_baseline', 'total_parameters': 107726601, 'trainable_parameters': 107726601, 'train_seconds': None, 'peak_gpu_gib': None, 'precision': 0.4732394366197183, 'recall': 0.5562913907284768, 'entity_f1': 0.5114155251141551}


### 4B. Full fine-tuning and evaluation

Same procedure and learning rate as for IMDb. The model is larger (BERT-base, about 108 million parameters, against 67 million for DistilBERT), so training takes longer.

In [13]:
ner_results.append(run_experiment(
    "ner_full", AutoModelForTokenClassification.from_pretrained(NER_MODEL),
    ner_tokenizer, ner_train_tok, ner_val_tok, ner_collator, ner_metrics,
    learning_rate=2e-5, train=True))
display(pd.DataFrame(ner_results))

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss


{'variant': 'ner_full', 'total_parameters': 107726601, 'trainable_parameters': 107726601, 'train_seconds': 4.4608956269999, 'peak_gpu_gib': 2.0340847969055176, 'precision': 0.520891364902507, 'recall': 0.6192052980132451, 'entity_f1': 0.5658093797276854}


,variant,total_parameters,trainable_parameters,train_seconds,peak_gpu_gib,precision,recall,entity_f1
0,ner_baseline,107726601,107726601,NaN,NaN,0.473239,0.556291,0.511416
1,ner_full,107726601,107726601,4.460896,2.034085,0.520891,0.619205,0.565809


### 4C. LoRA fine-tuning and evaluation

This task is token classification, so we use `TaskType.TOKEN_CLS`, and BERT names its attention projections `query` and `value`. `make_lora` loads a fresh copy of the original checkpoint: this run does not start from the model trained in 4B.

In [14]:
ner_results.append(run_experiment(
    "ner_lora", make_lora(NER_MODEL, AutoModelForTokenClassification,
                            TaskType.TOKEN_CLS, ["query", "value"]),
    ner_tokenizer, ner_train_tok, ner_val_tok, ner_collator, ner_metrics,
    learning_rate=2e-4, train=True))
display(pd.DataFrame(ner_results))

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


trainable params: 301,833 || all params: 108,028,434 || trainable%: 0.2794


Step,Training Loss


{'variant': 'ner_lora', 'total_parameters': 108028434, 'trainable_parameters': 301833, 'train_seconds': 2.6450802729996212, 'peak_gpu_gib': 0.6476473808288574, 'precision': 0.44327176781002636, 'recall': 0.5562913907284768, 'entity_f1': 0.49339207048458145}


,variant,total_parameters,trainable_parameters,train_seconds,peak_gpu_gib,precision,recall,entity_f1
0,ner_baseline,107726601,107726601,NaN,NaN,0.473239,0.556291,0.511416
1,ner_full,107726601,107726601,4.460896,2.034085,0.520891,0.619205,0.565809
2,ner_lora,108028434,301833,2.645080,0.647647,0.443272,0.556291,0.493392


**Questions:**

1. Compare entity F1 and costs across the three rows.
- **entity F1 was the highest with NER full-finetuning at 0.566, with the baseline NER coming in second at 0.511, and NER LoRA tokenization coming in last at 0.493. The cost (assuming as training time and peak GPU GiB) was most expensive with the NER full-finetuning at 4.46sec and 2.03 GiB peak GPU, whereas NER LoRA saved time to 2.65sec and save GPU memory with a peak of 0.65 GiB. The best at reducing cost was the baseline NER model since it does not take any time or GPU to train (since its pretrained).**

2. Most words of a sentence are tagged `O`. Why would the share of correctly tagged words (token accuracy) give an overly optimistic picture of the model?
- **It makes sense taht most words of a sentence are 0 since there generally are more filler words than persons' name in a sentence. This gives the impression that since the NER model is *correctly* labeling of these words as 0 means that the model is performant. However, those labeling could be obvious and simple, and not entirely helpful at determing more challenging labels like names/places/etc (ex. Claude a person vs Claude the AI model)**

3. What happens to an entity that lies after the 128-token cut?
- **The rest of their content gets left out since we are reducing reviews to fit within 128 tokens.**

4. WikiAnn tags were produced automatically from Wikipedia links, and some are wrong. How does this affect the interpretation of your scores?
- **In the example provided above `Barack/B-PER Obama/I-PER visited/O Paris/B-LOC`, there could be a case made that Paris is not a location but instead a person, and that would be a correct sentence but an incorrect label. This affects the interpretation of the scores since misclassification could be deteriorating the model's performance rather than the prediction method itself.**

## 5. Emotion classification (autonomous work)

Apply the same protocol to a new classification task: predict the emotion of a short message among **sadness, joy, love, anger, fear, surprise**.

- **Checkpoint:** `bhadresh-savani/distilbert-base-uncased-emotion`, a DistilBERT model that already predicts these six classes. It was fine-tuned on a dataset of Twitter messages ([model card](https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion)).
- **Dataset:** [GoEmotions](https://huggingface.co/datasets/SetFit/go_emotions), a different corpus of Reddit comments.

GoEmotions has 28 emotion categories, including `neutral`, and a comment can have several of them. The model expects exactly one of its six classes. The preparation cell below therefore keeps only the comments that have **a single label in total**, provided this label is one of the six emotions. A comment labelled `joy` and `surprise`, or only `neutral`, is removed. The label is then converted to the model's class number. The training and validation samples come from the separate train and validation splits of GoEmotions, and all three variants use the same samples.

Steps:

1. Run the preparation cell and read the sample sizes and the number of examples per class.
2. **TODO 1.** Tokenize the text as in the IMDb example, keeping at most `MAX_LENGTH` tokens, and create a `DataCollatorWithPadding`.
3. **TODO 2.** Evaluate the unchanged model (baseline) on the validation sample with `classification_metrics`.
4. **TODO 3.** Load a fresh copy of the model, fine-tune all its parameters on the training sample with `learning_rate=2e-5`, and evaluate it on the same validation sample.
5. **TODO 4.** Create a LoRA model with `make_lora` (`TaskType.SEQ_CLS`, DistilBERT layers `q_lin` and `v_lin`), train it on the same sample with `learning_rate=2e-4`, and evaluate it.
6. **TODO 5.** Display a table comparing accuracy, macro F1, trainable parameters, time and GPU memory. Explain your results, including the effect of class imbalance. Additional fine-tuning can also lower a score.

Reuse the functions defined above. Each `TODO` must end with a cell that runs and prints a result. If training is too slow, you may lower `TRAIN_SIZE` or `VALID_SIZE`, then rerun the preparation cell and all three variants with the new sizes.

In [15]:
EMOTION_MODEL = "bhadresh-savani/distilbert-base-uncased-emotion"
emotions = load_dataset("SetFit/go_emotions")
emotion_tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL)
emotion_probe = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL)
emotion_label_to_id = {name.lower(): int(idx) for name, idx in emotion_probe.config.label2id.items()}
print("Model labels:", emotion_label_to_id)
# In this version of GoEmotions, each emotion is a column containing 0 or 1.
all_emotions = [name for name in emotions["train"].column_names if name != "text"]
assert set(emotion_label_to_id).issubset(all_emotions)
del emotion_probe
gc.collect()

def prepare_emotions(split):
    """Keep comments with a single label among the six emotions; store the model's class number."""
    one_supported = split.filter(lambda row: sum(row[name] for name in all_emotions) == 1
                                 and any(row[name] for name in emotion_label_to_id))
    def add_label(row):
        name = next(name for name in emotion_label_to_id if row[name] == 1)
        return {"label": emotion_label_to_id[name]}
    return one_supported.map(add_label, remove_columns=all_emotions)

emotion_train = sample_split(prepare_emotions(emotions["train"]), TRAIN_SIZE)
emotion_val = sample_split(prepare_emotions(emotions["validation"]), VALID_SIZE)
id_to_emotion = {idx: name for name, idx in emotion_label_to_id.items()}
print("Sample sizes:", len(emotion_train), len(emotion_val))
print("Training examples per class:")
print(pd.Series(list(emotion_train["label"])).map(id_to_emotion).value_counts())
print("Validation examples per class:")
print(pd.Series(list(emotion_val["label"])).map(id_to_emotion).value_counts())
print("Example:", emotion_train[0])

README.md:   0%|          | 0.00/236 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/19.2M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

... (9 outputs left out)

Map:   0%|          | 0/5272 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/625 [00:00<?, ? examples/s]

Sample sizes: 400 200
Training examples per class:
love        117
anger        77
joy          73
surprise     52
sadness      45
fear         36
Name: count, dtype: int64
Validation examples per class:
love        64
anger       36
joy         31
sadness     28
surprise    27
fear        14
Name: count, dtype: int64
Example: {'text': 'i had all of those. checked myself with a GP and found out i had low iron, migraines and needed to visit the therapist re my anxiety.', 'label': 4}


In [ ]:
# TODO 1: tokenize emotion_train and emotion_val with emotion_tokenizer.
# Write a function similar to tokenize_reviews, then use .map(..., batched=True).
# Keep the "label" column and remove "text". Create emotion_collator.

def tokenize_emotions(batch):
    """Convert emotions to token IDs, keeping at most MAX_LENGTH tokens (special tokens included)."""
    return emotion_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

emotion_train_tok = emotion_train.map(tokenize_emotions, batched=True)
emotion_val_tok = emotion_val.map(tokenize_emotions, batched=True)
emotion_collator = DataCollatorWithPadding(tokenizer=emotion_tokenizer)

# imdb_collator = DataCollatorWithPadding(tokenizer=imdb_tokenizer)
# imdb_train_tok = imdb_train.map(tokenize_reviews, batched=True, remove_columns=["text"])

In [ ]:
emotion_results = []
# TODO 2: evaluate a fresh copy of EMOTION_MODEL with run_experiment (train=False).
# Append the returned dictionary to emotion_results.
emotion_results.append(run_experiment(
    "emotion_baseline", AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL),
    emotion_tokenizer, emotion_train_tok, emotion_val_tok, emotion_collator, classification_metrics,
    learning_rate=2e-5, train=False
))

In [ ]:
# TODO 3: fine-tune a fresh copy of EMOTION_MODEL on emotion_train_tok
# (all parameters, learning_rate=2e-5, train=True) and append its result.

In [ ]:
# TODO 4: create a LoRA model with make_lora, train it (learning_rate=2e-4, train=True)
# and append its result to emotion_results.

In [ ]:
# TODO 5: display your comparison table, then answer the questions below.
# display(pd.DataFrame(emotion_results))

### Written answers

1. How do validation accuracy and macro F1 differ across the three variants? Which classes are rare in your sample, and how do they affect macro F1?
2. Compare trainable parameters, training time and peak GPU memory. Do your measurements allow you to say that one method is always faster? Why?
3. Why should the final score be measured on a separate test set once a method or a learning rate has been chosen with the validation set?
4. The checkpoint was trained on Twitter messages and we evaluate it on Reddit comments. How can this change of source affect its predictions? What do we lose by removing the GoEmotions comments that have several labels?
5. In TP2, the sentiment pipeline gave each review a signed score. Explain what this score measures, and how it differs from a LoRA weight and from the training loss.

**Submission:** the notebook with the autonomous cells completed, the printed results, the comparison tables and your written answers. If a run fails, keep the error message in the notebook and explain which step you could not complete.